In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
from numpyro.contrib.nested_sampling import NestedSampler

C:\Users\honza\AppData\Roaming\Python\Python313\site-packages\jaxns\internals\mixed_precision.py:14: UserWarning: JAX x64 is not enabled. Setting it now. Check for errors.
  warnings.warn("JAX x64 is not enabled. Setting it now. Check for errors.")
INFO:2026-02-25 22:41:00,125:jax._src.xla_bridge:834: Unable to initialize backend 'tpu': UNIMPLEMENTED: LoadPjrtPlugin is not implemented on windows yet.
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': UNIMPLEMENTED: LoadPjrtPlugin is not implemented on windows yet.


AttributeError: module 'jax.interpreters.xla' has no attribute 'pytype_aval_mappings'

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────

def gauss_jax(x, k, mu, fwhm):
    sig = fwhm / 3e5 * mu / 2.35482
    return k * jnp.exp(-((x - mu) ** 2) / (2.0 * sig * sig))

def powerlaw(x, amplitude, x_0, alpha):
    return amplitude * (x / x_0) ** (-alpha)

def Halpha_jax(x, z, cont, cont_grad, Hal_peak, NII_peak, Nar_fwhm, SII_rpk, SII_bpk):
    Hal_wv = 6564.52 * (1 + z) / 1e4
    NII_r  = 6585.27 * (1 + z) / 1e4
    NII_b  = 6549.86 * (1 + z) / 1e4
    SII_r  = 6732.67 * (1 + z) / 1e4
    SII_b  = 6718.29 * (1 + z) / 1e4



    return (
        powerlaw(x, cont, Hal_wv, cont_grad)
        + gauss_jax(x, Hal_peak,      Hal_wv, Nar_fwhm)
        + gauss_jax(x, NII_peak,      NII_r,  Nar_fwhm)
        + gauss_jax(x, NII_peak / 3,  NII_b,  Nar_fwhm)
        + gauss_jax(x, SII_rpk,       SII_r,  Nar_fwhm)
        + gauss_jax(x, SII_bpk,       SII_b,  Nar_fwhm)
    )

def dict_entry_to_numpyro_dist(entry):
    """
    Converts a prior dict entry to a numpyro distribution.

    Supported kinds
    ---------------
    normal      : [init, 'normal',      mu,    sigma        ]
    uniform     : [init, 'uniform',     lo,    hi           ]
    loguniform  : [init, 'loguniform',  lo_l10, hi_l10      ]  bounds in log10
    halfnormal  : [init, 'halfnormal',  _,     sigma        ]
    truncnormal : [init, 'truncnormal', mu,    sigma, lo, hi]
    """
    _, kind, a, b = entry[:4]
    kind = kind.lower()

    if kind == 'normal':
        return dist.Normal(float(a), float(b))

    elif kind == 'uniform':
        return dist.Uniform(float(a), float(b))

    elif kind == 'loguniform':
        # a, b are log10 bounds → TransformedDistribution: 10^Uniform(a,b)
        return dist.TransformedDistribution(
            dist.Uniform(float(a), float(b)),
            dist.transforms.AffineTransform(0, 1)   # identity; exponentiation below
        )
        # Note: handled explicitly in the model via 10** — see numpyro_model()

    elif kind == 'halfnormal':
        return dist.HalfNormal(float(b))

    elif kind == 'truncnormal':
        _, kind, mu, sigma, lo, hi = entry
        return dist.TruncatedNormal(float(mu), float(sigma),
                                    low=float(lo), high=float(hi))
    else:
        raise ValueError(f"Unknown prior kind: '{kind}'")

# ── numpyro model ─────────────────────────────────────────────────────────────

PARAM_NAMES = ['z', 'cont', 'cont_grad', 'Hal_peak', 'NII_peak',
               'Nar_fwhm', 'SII_rpk', 'SII_bpk']

priors = {
    'z':         [1,    'normal',     1,    0.003],
    'cont':      [0.1,  'loguniform', -4,   1    ],
    'cont_grad': [-1,   'normal',     0,    0.3  ],
    'Hal_peak':  [0.5,  'loguniform', -3,   1    ],
    'NII_peak':  [0.6,  'loguniform', -3,   1    ],
    'Nar_fwhm':  [300,  'uniform',    100,  900  ],
    'SII_rpk':   [0.1,  'loguniform', -3,   1    ],
    'SII_bpk':   [0.1,  'loguniform', -3,   1    ],
}

def numpyro_model(x, y, yerr, priors):
    params = {}
    for name in PARAM_NAMES:
        entry = priors[name]
        kind  = entry[1].lower()
        a, b  = entry[2], entry[3]

        if kind == 'loguniform':
            # Sample in log10-space, then exponentiate
            log_val     = numpyro.sample(f'{name}_log10', dist.Uniform(float(a), float(b)))
            params[name] = 10.0 ** log_val
        elif kind == 'normal':
            params[name] = numpyro.sample(name, dist.Normal(float(a), float(b)))
        elif kind == 'uniform':
            params[name] = numpyro.sample(name, dist.Uniform(float(a), float(b)))
        elif kind == 'halfnormal':
            params[name] = numpyro.sample(name, dist.HalfNormal(float(b)))
        elif kind == 'truncnormal':
            mu, sigma, lo, hi = entry[2], entry[3], entry[4], entry[5]
            params[name] = numpyro.sample(
                name, dist.TruncatedNormal(float(mu), float(sigma),
                                           low=float(lo), high=float(hi)))
        else:
            raise ValueError(f"Unknown prior kind: '{kind}'")

    model_flux = Halpha_jax(x, **params)
    numpyro.sample('obs', dist.Normal(model_flux, yerr), obs=y)

# ── Run nested sampling ───────────────────────────────────────────────────────

def run_nested(x_data, y_data, yerr_data, priors,
               num_live_points=500, max_samples=100_000, rng_seed=42):

    x    = jnp.asarray(x_data,    dtype=jnp.float32)
    y    = jnp.asarray(y_data,    dtype=jnp.float32)
    yerr = jnp.asarray(yerr_data, dtype=jnp.float32)

    ns = NestedSampler(
        numpyro_model,
        constructor_kwargs=dict(num_live_points=num_live_points,
                                max_samples=max_samples)
    )

    ns.run(jax.random.PRNGKey(rng_seed), x, y, yerr, priors)
    ns.print_summary()

    samples = ns.get_samples(jax.random.PRNGKey(1), num_samples=5000)
    return samples

# ── Diagnostics ───────────────────────────────────────────────────────────────

def print_summary(samples):
    print(f"\n{'Parameter':<14} {'Median':>10} {'+ err':>10} {'- err':>10}")
    print("-" * 48)
    for name in PARAM_NAMES:
        key = f'{name}_log10' if priors[name][1].lower() == 'loguniform' else name
        if key in samples:
            s = np.array(samples[key])
            if priors[name][1].lower() == 'loguniform':
                s = 10.0 ** s          # convert back to physical
        lo, med, hi = np.percentile(s, [16, 50, 84])
        print(f"  {name:<12}  {med:>10.4f}  +{hi - med:.4f}  -{med - lo:.4f}")

# ── Entry point ───────────────────────────────────────────────────────────────



In [ ]:
# ── Priors dict — edit freely, changes propagate automatically ────────────────

priors = {
    'z':         [1,    'normal',     1,    0.003],
    'cont':      [0.1,  'loguniform', -4,   1    ],
    'cont_grad': [-1,   'normal',     0,    0.3  ],
    'Hal_peak':  [0.5,  'loguniform', -3,   1    ],
    'NII_peak':  [0.6,  'loguniform', -3,   1    ],
    'Nar_fwhm':  [300,  'uniform',    100,  900  ],
    'SII_rpk':   [0.1,  'loguniform', -3,   1    ],
    'SII_bpk':   [0.1,  'loguniform', -3,   1    ],
}

# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    x_data    = np.linspace(0.87, 0.92, 300, dtype=np.float32)
    y_data    = np.ones(300, dtype=np.float32) * 0.05
    yerr_data = np.ones(300, dtype=np.float32) * 0.01

    results = run_nested(x_data, y_data, yerr_data, priors,
                         num_live_points=500, max_samples=200_000)
    diagnostics(results, priors)